In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import ExponentialSmoothing, SimpleExpSmoothing, Holt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.forecasting.theta import ThetaModel
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from os import walk
pd.options.plotting.backend = "plotly"

# Functions


In [ ]:
def read_by_mask(path, mask, **args):
    filenames = next(walk(path), (None, None, []))[2]  # [] if no file
    df = pd.DataFrame()
    for xlsx in filenames:
        if mask in xlsx:
            df = pd.concat([df, pd.read_excel(path+xlsx, **args)])
    return df

# import data (early precalculated) from yandex-disk

import requests
from urllib.parse import urlencode

def read_csv_from_yd(public_key, **args):
  '''public_key - ссылка с доступом на скачивание файла
  '''
  base_url = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?'

  # Получаем загрузочную ссылку
  final_url = base_url + urlencode(dict(public_key=public_key))
  response = requests.get(final_url)
  download_url = response.json()['href']


  return pd.read_csv(download_url, **args)

def read_xlsx_from_yd(public_key, **args):
  '''public_key - ссылка с доступом на скачивание файла
  '''
  base_url = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?'

  # Получаем загрузочную ссылку
  final_url = base_url + urlencode(dict(public_key=public_key))
  response = requests.get(final_url)
  download_url = response.json()['href']


  return pd.read_excel(download_url)

In [ ]:
def build_density_forecast(abt, pair, horizon, base_alg = ExponentialSmoothing,
                           base_params = {'seasonal_periods':365,
                                        'seasonal':"add",
                                        "initialization_method":"estimated"},
                           omega_size = None,
                           bins = 'auto', fcst_id = -1, d_forecast = None):

    all_days = pd.date_range(abt.index.min(), abt.index.max(), freq='D')

    ts = abt[(abt['product_id'] == pair[0]) & (abt['location_id'] == pair[1])][['s_qty']]\
        .reindex(all_days, fill_value=0).fillna(0)

    if d_forecast is not None:
        d_forecast=d_forecast[(d_forecast['product_id'] == pair[0]) & (d_forecast['location_id'] == pair[1])]

    if omega_size is None or omega_step<1:
        omega_size = int(max(10, min(100, ts.s_qty.max()- ts.s_qty.min())))

    # build forecast
    fcst_density =  get_density_forecast(ts, horizon, base_alg=base_alg,
            base_params={'seasonal_periods':365, 'seasonal':"add", "initialization_method":"estimated"},
                omega=list(np.linspace(0, max(ts.s_qty.max(),1.0), omega_size))
                , fittedvalues=False
                , _low = 0
                , d_forecast = d_forecast)

    if fcst_id>0:
        plot_density_forecast(fcst_density, ts, fcst_id)

    return fcst_density

def get_density_forecast(ts, horizon, base_alg, base_params={}, omega=None, fittedvalues=False,
                         _low = np.nan, d_forecast = None):
    """
    Returns a list of density dictionaries {'bins': np.array, 'probs': np.array, 'dotted_forecast': float}.

    Parameters
    ----------
    ts : array_like
        The time series to model.
    horizon : int
        The horizon to forecast.
    base_alg : {ExponentialSmoothing, SimpleExpSmoothing, Holt}
        The name of base algoritm for making density forecast.
    base_params : dict
        A Dictionary with base algorithm parameters.
    bins: int or sequence of scalars or str, optional
        Define how to calculate bins.
    fittedvalues: bool
        Include fitted values in density dictionaries or not.
    """

    if omega is not None:
        bins = omega


    alg = base_alg(ts, **base_params).fit()

    if fittedvalues:
        alg_preds = alg.predict(start=0, end=len(ts) + horizon - 1).clip(lower = _low)
        density_dicts = [{'bins': [], 'probs': [], 'dotted_forecast': None} for _ in range(len(ts) + horizon)]
    else:
        alg_preds = alg.predict(start=len(ts), end=len(ts) + horizon - 1).clip(lower = _low)
        density_dicts = [{'bins': [], 'probs': [], 'dotted_forecast': None} for _ in range(horizon)]

#     print(d_forecast)

    if d_forecast is not None:
        altern_preds = d_forecast.merge(alg_preds.rename('auto'),
                                        how = 'right', left_index = True, right_index = True)
        alg_preds = altern_preds['forecast_qty'].combine_first(altern_preds['auto'])

    for i in range(len(alg_preds)):
        density_dicts[i]['dotted_forecast'] = alg_preds.iloc[i]
        density_dicts[i]['period_dt']  = alg_preds.index[i]
        current_density = alg.resid + alg_preds.iloc[i]
        probs, bins = np.histogram(current_density, bins=bins)

        density_dicts[i]['probs'], density_dicts[i]['bins'] = probs / np.sum(probs), bins

    return density_dicts

def plot_density_forecast(fcst_density, ts, fcst_id = 1, ax=None, **kwargs):

    trace1 = go.Bar( y=list(fcst_density[fcst_id]['bins'][:-1]), x=list(fcst_density[fcst_id]['probs']),
            name="Гистограмма вероятности n продаж", orientation='h')

    trace12 = go.Bar( y=list(fcst_density[fcst_id]['bins'][:-1]),
            x=[ fcst_density[fcst_id]['probs'][i]*(fcst_density[fcst_id]['bins'][i]<=fcst_density[fcst_id]['dotted_forecast']
             and fcst_density[fcst_id]['bins'][i+1]>fcst_density[fcst_id]['dotted_forecast'])
             for i in range(len(fcst_density[fcst_id]['probs']))]
          ,name='Вероятность точечного прогноза', orientation='h'
      )


    trace2 = go.Scatter(
        x=list(ts.index),
        y=list(ts.s_qty),
        name='Time Series'
    )

    trace3 = go.Scatter(
      x=[f['period_dt'] for f in fcst_density] # list(fr[f['dotted_forecast'] for f in fcst_density] # c.index)
      ,y= [f['dotted_forecast'] for f in fcst_density] # list(frc.Forecast)
      ,name='Forecast'
      )

    trace4 = go.Scatter(
        x=[fcst_density[fcst_id]['period_dt'], fcst_density[fcst_id]['period_dt']],
        y=[fcst_density[fcst_id]['dotted_forecast'],fcst_density[fcst_id]['dotted_forecast']],
        name='Dotter Forecast'
    )


    fig = make_subplots(rows=1, cols=5, shared_xaxes=True,
        specs=[[{"colspan": 4}, None, None, None, {}],
              ],
        print_grid=False)

    fig.append_trace(trace3, 1,1)
    fig.append_trace(trace2, 1,1)
    fig.append_trace(trace4, 1,1)
    fig.append_trace(trace1, 1,5)
    fig.append_trace(trace12, 1,5)


    fig['layout'].update(height=400, width=1000)
    fig.show()

    return

In [ ]:
def purchase_sh(PBR, n = 10):
    '''Моделируем график поставок на склад при заданной частоте поставок'''

    if PBR<=0:
        return list([1]*n)
    else:
        if PBR == 1:
            return [1,0,0,0,0,0,0]*int(np.ceil(n/7))
        if PBR ==2:
#             print('ok')
            return [1,0,0,1,0,0,0]*int(np.ceil(n/7))
        if PBR ==3:
            return [1,0,1,0,1,0,0]*int(np.ceil(n/7))
        if PBR ==4:
            return [1,0,1,0,1,0,1]*int(np.ceil(n/7))
        if PBR ==5:
            return [1,0,1,1,1,0,1]*int(np.ceil(n/7))
        if PBR ==6:
            return [1,1,1,0,1,1,1]*int(np.ceil(n/7))
        if PBR ==7:
            return [1,1,1,1,1,1,1]*int(np.ceil(n/7))


In [ ]:
def dict_to_df(df):
    return pd.DataFrame.from_dict({'bins':df['bins'][:-1], 'probs':df['probs']})

def agg_density_forecast(df1, df2):
    _df1 = dict_to_df(df1)
    _df2 = dict_to_df(df2)

    _df = _df1.merge(_df2, how = 'cross')
    _df['bins'] = _df[['bins_x', 'bins_y']].sum(axis = 1)
    _df['probs'] = _df[['probs_x', 'probs_y']].prod(axis = 1)
    _df = _df.groupby('bins').sum()[['probs']].reset_index()

    return {'bins': list(_df.bins)+[df1['bins'][-1]+df2['bins'][-1]], 'probs':_df.probs}

def reshape_barchart(df, newbins=None, step = 1):
    if not newbins:
        newbins = list(range(0, int(np.ceil(df['bins'][-1]/step)*step)+1, step))

    # create ranges for new probs calculation
    _df = pd.DataFrame.from_dict({'bins':concat_bins(newbins, df['bins'])})\
        .merge(pd.DataFrame({'old_bins':df['bins'][:-1],'old_ends':df['bins'][1:],'probs':df['probs']}),
                            how='left', left_on = 'bins', right_on = 'old_bins')\
        .merge(pd.DataFrame({'new_bins':newbins[:-1],'new_ends':newbins[1:]}),
                          how = 'left', left_on = 'bins', right_on = 'new_bins').ffill()
    _df['new_probs'] = (_df['probs']*
            (_df[['new_ends','old_ends']].min(axis=1)-_df[['new_bins','old_bins']].max(axis=1))/
             (_df['old_ends']-_df['old_bins']))

    # calculate new probs
    final_df = _df[['new_probs','new_bins']].groupby('new_bins').sum().reset_index()

    return {'bins':list(final_df.new_bins)+[df['bins'][-1]],'probs':final_df.new_probs}


def get_percentile(df, q=0.5, strict = False):
    '''df - density forecast dictionary'''
    q = min(1,max(0,q))

    cumulatice_sum = np.cumsum([0]+list(df['probs'])+[0])
    idx = cumulatice_sum>=q
    idx = (idx[:-1] | idx[1:])

    quantiles = np.array(df['bins'])[idx][:2]
    probs = cumulatice_sum[:-1][idx][:2]

    if strict:
        return quantiles[0]+ (q-probs[0])/(probs[1]-probs[0])*(quantiles[1]-quantiles[0]), probs, quantiles
    else:
        return quantiles[1], probs, quantiles

def concat_bins(bins1, bins2):
    l = []
    i1 = 0
    i2 = 0

    while i1+i2 < (len(bins1)+len(bins2)):
        if bins1[i1]<bins2[i2]:
            l += [bins1[i1]]
            i1+=1
        else:
            l += [bins2[i2]]
            if bins1[i1]==bins2[i2]:
                i1+=1
            i2 += 1

        if i1 == len(bins1):
            l = l+bins2[i2:]
            i2 = len(bins2)
#             print(i)

        if i2 == len(bins2):
            l = l+bins1[i1:]
            i1 = len(bins1)
#             print(i)
    return l


def calculate_norms(pair, abt, fcst_density, lt, sl, pipeline = None):
    '''Расчёт норм в динамике
    Input:
    pair - np.array из двух элементов: np.array([product_id, location_id])
    abt - pd.DataFrame(columns = ['product_id', 'location_id', 'Stock','прогноз'], index = 'period_dt')
    fcst_density - np.array(dict('bins':[n], 'probs':list([n-1]), 'period_dt':),
    lt - pd.DataFrame(columns = ['product_id', 'location_id', 'LT', 'Квант','MerchMinimum','PBR'])
    sl - pd.DataFrame([])
    pipeline - pd.DataFrame([])


    Output

    '''
    all_days = pd.date_range(abt.index.min(), abt.index.max(), freq='D')

    norms = abt[(abt['product_id'] == pair[0]) & (abt['location_id'] == pair[1])]\
    .reindex(all_days).ffill()[['product_id', 'location_id', 'stock']].tail(1)

    fut_days = pd.date_range(norms.index.max(), fcst_density[-1]['period_dt'], freq='D')
    norms = norms.reindex(fut_days, fill_value=np.nan).ffill()


    # считываем гиперпараметры
    _lt = lt[(lt['product_id'] == pair[0])& (lt['location_id'] == pair[1])].LT.values[0]
    _sl = sl[(sl['product_id'] == pair[0]) & (sl['location_id'] == pair[1])].SL.values[0]
    _quant = lt[(lt['product_id'] == pair[0]) & (lt['location_id'] == pair[1])]['Квант'].values[0]
    _mm = lt[(lt['product_id'] == pair[0]) & (lt['location_id'] == pair[1])]['MerchMinimum'].values[0]
    PBR = lt[(lt['product_id'] == pair[0]) & (lt['location_id'] == pair[1])]['PBR'].values[0]  # частота поставки

    _PBR = int(np.ceil(7/max(int(str(PBR).replace('-1','7').replace('0','7')),1))) # кол-во дней между заказами (period between replenishment)

    # флаг того, что нужно заказ делать по расписанию (а не когда закончится)
    BS_policy = lt[(lt['product_id'] == pair[0]) & (lt['location_id'] == pair[1])]['BS_policy'].values[0]

    # расчётные показатели
    norms['дата заказа'] = purchase_sh(PBR, len(fut_days))[:len(fut_days)]
    norms['прогноз'] = [0] + [f['dotted_forecast'] for f in fcst_density]
    norms['норма']= np.nan
    norms['точка заказа']= np.nan
    norms['квант sl']= np.nan
    norms['пред квант sl']= np.nan
    norms['чистая норма']= np.nan
    norms['квант норма']= np.nan
    norms['квант']= _quant
    norms['мерчминимум']= _mm
    norms['в пути']= 0
    norms['СЗ']= 0


    # инициализируем стоки и товары в пути на момент расчёта

    stock_dyn = (norms.iloc[0, norms.columns.get_loc('stock')]
                - norms.iloc[0:_lt, norms.columns.get_loc('прогноз')].cumsum()).to_frame(name = 'stock')

    if pipeline is not None:
        stock_dyn = stock_dyn.merge(pipeline[(pipeline['product_id'] == pair[0]) & (pipeline['location_to_id'] == pair[1])].set_index('receive_date'),
                how='left', left_index = True, right_index = True).ffill().ffill(0)[['stock', 'volume']]
    else:
        stock_dyn['volume'] = 0
        stock_dyn.iloc[0, stock_dyn.columns.get_loc('volume')] = np.ceil(max(0, -stock_dyn.min().values[0])/_quant)*_quant

    norms.iloc[1:_lt+1, norms.columns.get_loc('stock')] = stock_dyn.sum(axis = 1)
    norms.iloc[_lt+1:, norms.columns.get_loc('stock')] = 0

    norms.iloc[1:_lt+1, norms.columns.get_loc('в пути')]= stock_dyn.volume

#     print(_quant, _sl)
    for d in range(len(fut_days[:-(_lt+_PBR)])):
        if norms.iloc[d]['дата заказа']==1:
            new_df = {'bins':[0, 0], 'probs':[1]}
            for s in range(_lt, _lt + _PBR, 1):
                    # здесь грубо считается период покрытия: нужно смотреть на реальную дату следующего заказа
                new_df = agg_density_forecast(new_df, fcst_density[d+s])

            pre_norma = get_percentile(reshape_barchart(new_df, step=1), _sl, )[0]
            norms.iloc[d+_lt:d+_lt+_PBR,norms.columns.get_loc('чистая норма')] = pre_norma
            norma = max(pre_norma, _mm)
            norms.iloc[d+_lt:d+_lt+_PBR,norms.columns.get_loc('норма')] = norma

            q_norma, probs, quantiles = get_percentile(reshape_barchart(new_df, step=int(_quant)), _sl)
            norms.iloc[d+_lt:d+_lt+_PBR,norms.columns.get_loc('квант норма')] = q_norma
            norms.iloc[d+_lt:d+_lt+_PBR,norms.columns.get_loc('квант sl')]= probs[1]
            norms.iloc[d+_lt:d+_lt+_PBR,norms.columns.get_loc('пред квант sl')]= probs[0]
            reorder_level = max(0, norma-1) if BS_policy else 0
            norms.iloc[d+_lt:d+_lt+_PBR,norms.columns.get_loc('точка заказа')] = reorder_level

            norms.iloc[d+_lt:d+_lt+_PBR,norms.columns.get_loc('СЗ')] = ( norma -
                            norms.iloc[d+_lt:d+_lt+_PBR,norms.columns.get_loc('прогноз')].sum())

    #         if fut_days[d]==pd.Timestamp('2022-06-05 00:00:00'):
    #             print(order, norma, norms.iloc[d+_lt, norms.columns.get_loc('stock')])

            order = np.ceil(1.0*max(norma - norms.iloc[d+_lt]['stock'],0)/_quant)*_quant if norms.iloc[d+_lt]['stock']<=reorder_level else 0

            norms.iloc[d+_lt,norms.columns.get_loc('в пути')] = norms.iloc[d+_lt]['в пути']+order
            norms.iloc[d+_lt, norms.columns.get_loc('stock')] += order

    #         if fut_days[d]==pd.Timestamp('2022-06-05 00:00:00'):
    #             print(order, norma, norms.iloc[d+_lt, norms.columns.get_loc('stock')])

        stock = norms.iloc[d+_lt]['stock'] - norms.iloc[d+_lt]['прогноз'] #+ norms.iloc[d+_lt]['в пути']
        norms.iloc[d+_lt+1, norms.columns.get_loc('stock')] =  max(0, stock)
    return norms

In [ ]:
def qualityWAMAXPE(x,y):
    # Weighted absolute maximum percentage error
    # x,y - pandas structures
    # x - real values
    # y - forecasts
    denom = pd.merge(x, y, right_index = True, left_index = True).max(axis = 1).sum()
    qlt = ((x-y).abs()/denom).replace([np.inf, -np.inf], np.nan)
    return qlt.sum() , qlt

In [ ]:
# path = '/home/aromanenko/Yandex.Disk/Consulting/StroyDvor/IO/data/data_may22/'

# abt = read_by_mask(path, 'ABT', date_parse = ['period_dt'], dayfirst = True).set_index('period_dt')
# d_forecast = read_by_mask(path, 'forecast', date_parse = ['period_dt'], dayfirst = True).set_index('period_dt')
# pipeline = read_by_mask(path, 'pipeline', date_parse = True, dayfirst = True)

In [ ]:
# horizon = 100
# all_days = pd.date_range(abt.index.min(), abt.index.max(), freq='D')
# # train forecasting models

# ts = abt[(abt['product_id'] == _id[2][0]) & (abt['location_id'] == _id[2][1])][['s_qty']]\
#         .reindex(all_days, fill_value=0).fillna(0)

# agg_ts = ts[['s_qty']].resample('M').sum()
# esm_ = ExponentialSmoothing(
#     agg_ts[['s_qty']],
#     seasonal_periods=12,
#     seasonal="add",
#     trend = 'add',
#     initialization_method="estimated"
#     ).fit()



# # build forecast
# frc =  esm_.predict(start=0, end=len(agg_ts) + horizon - 1).clip(lower = 0.).to_frame(name = 'Forecast')

# # Quality within last 3 months
# start_dt = '01-01-2022'
# end_dt = '31-03-2022'
# ix = pd.date_range(pd.to_datetime(start_dt), pd.to_datetime(end_dt), freq = 'M')
# WAPE,_ = qualityWAMAXPE(agg_ts['s_qty'].loc[ix].resample('M').sum(), frc.loc[ix]['Forecast'].resample('M').sum())


# fig = frc.merge(agg_ts, how='left', left_index = True, right_index = True)\
# .plot(title = 'product = {0}, location = {1}, Ошибка WAPE = {2}%'.format(i[0], i[1], np.floor(WAPE*100)))
# fig.update_layout(autosize=False,width=1000,height=400,).show()

In [ ]:
# _id = [[2777, 1032], [233204, 1025], [56012, 1025], [5581, 1025], [7348, 1021]]

# for i in _id:
#     # data preprocess before forecasting
#     ts = abt[(abt['product_id'] == i[0]) & (abt['location_id'] == i[1])][['s_qty']]\
#         .reindex(all_days, fill_value=0).fillna(0)


#     # train forecasting models
#     esm_ = ExponentialSmoothing(
#         ts[['s_qty']],
#         seasonal_periods=365,
#         seasonal="add",
#         initialization_method="estimated"
#         ).fit()



#     # build forecast
#     frc =  esm_.predict(start=0, end=len(ts) + horizon - 1).clip(lower = 0.).to_frame(name = 'Forecast')

#     # Quality within last 3 months
#     start_dt = '01-01-2022'
#     end_dt = '31-03-2022'
#     ix = pd.date_range(pd.to_datetime(start_dt), pd.to_datetime(end_dt))
#     WAPE,_ = qualityWAMAXPE(ts['s_qty'].loc[ix].resample('M').sum(), frc.loc[ix]['Forecast'].resample('M').sum())


#     fig = frc.merge( ts, how='left', left_index = True, right_index = True)\
#     .plot(title = 'product = {0}, location = {1}, Ошибка WAPE = {2}%'.format(i[0], i[1], np.floor(WAPE*100)))
#     fig.update_layout(autosize=False,width=1000,height=400,).show()


# Моделирование внешных данных (Сервис 1)


In [ ]:
# здесь должна быть вычисление lt и оценка потенциальных дат заказов (еcли необходимо)
# lt = pd.read_csv('./data/lead_time.csv')
# lt.head()
# LT = lt.groupby('product_id').min()[['LT', 'Квант', 'MerchMinimum', 'PBR','BS_policy']]

# Уровень сервиса

In [ ]:
# sl = pd.read_csv('./data/service_lvl.csv')
# sl.head()
# SL = sl.groupby('product_id').min()[['SL']]

In [ ]:
# fig = make_subplots(cols = 5, rows = 1, shared_xaxes=True)

# trace1 = go.Bar( x=[str(l) for l in LT.index],y=LT.LT,name='Lead Times')
# trace2 = go.Bar( x=[str(l) for l in LT.index],y=LT['Квант'], name='Квант')
# trace3 = go.Bar( x=[str(l) for l in LT.index],y=LT['MerchMinimum'],name='Мерчминимум')
# trace4 = go.Bar( x=[str(l) for l in SL.index],y=SL['SL'],name='Service Level')
# trace5 = go.Bar( x=[str(l) for l in LT.index],y=LT['PBR'],name='Период между поставками')

# fig.append_trace(trace1, 1,1)
# fig.append_trace(trace2, 1,2)
# fig.append_trace(trace3, 1,3)
# fig.append_trace(trace4, 1,4)
# fig.append_trace(trace5#         print(fut_days[d])
# , 1,5)

# fig.update_layout(autosize=False,width=900,height=300,).show()

# Оптимальные нормы

In [ ]:
# horizon = 100
# num = 0
# fcst_density = build_density_forecast(abt, pair=_id[num], horizon = horizon, fcst_id = 90, omega_step = 5)
# norms = calculate_norms(_id[num], abt, fcst_density, lt, sl)
# norms[['stock', 'в пути', 'СЗ', 'норма', 'чистая норма','прогноз', 'квант норма', 'дата заказа']]\
#     .plot(title = 'product = {0} location = {1}<br>Service Level = {2}, PBR = {6}, Lead Time = {3}, \n Квант = {4}, \n МерчМинимум = {5}'\
#           .format(_id[num][0], _id[num][1],
#                   sl[(sl['product_id'] == _id[num][0])&(sl['location_id'] == _id[num][1])].SL.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].LT.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])]['Квант'].values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].MerchMinimum.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].PBR.values[0]))\
#     .update_layout(height=450, width=950)

In [ ]:
# horizon = 100
# num = 1
# fcst_density = build_density_forecast(abt, pair=_id[num], horizon = horizon, fcst_id = 90, omega_step = 5)
# norms = calculate_norms(_id[num], abt, fcst_density, lt, sl)
# norms[['stock', 'в пути', 'СЗ', 'норма', 'чистая норма','прогноз', 'квант норма', 'дата заказа']]\
#     .plot(title = 'product = {0} location = {1}<br>Service Level = {2}, PBR = {6}, Lead Time = {3}, \n Квант = {4}, \n МерчМинимум = {5}'\
#           .format(_id[num][0], _id[num][1],
#                   sl[(sl['product_id'] == _id[num][0])&(sl['location_id'] == _id[num][1])].SL.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].LT.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])]['Квант'].values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].MerchMinimum.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].PBR.values[0]))\
#     .update_layout(height=450, width=950)

In [ ]:
# horizon = 100
# num = 2
# fcst_density = build_density_forecast(abt, pair=_id[num], horizon = horizon, fcst_id = 90, omega_step = 5)
# norms = calculate_norms(_id[num], abt, fcst_density, lt, sl)
# norms[['stock', 'в пути', 'СЗ', 'норма', 'чистая норма','прогноз', 'квант норма', 'дата заказа']]\
#     .plot(title = 'product = {0} location = {1}<br>Service Level = {2}, PBR = {6}, Lead Time = {3}, \n Квант = {4}, \n МерчМинимум = {5}'\
#           .format(_id[num][0], _id[num][1],
#                   sl[(sl['product_id'] == _id[num][0])&(sl['location_id'] == _id[num][1])].SL.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].LT.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])]['Квант'].values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].MerchMinimum.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].PBR.values[0]))\
#     .update_layout(height=450, width=950)

In [ ]:
# horizon = 100
# num = 3
# fcst_density = build_density_forecast(abt, pair=_id[num], horizon = horizon, fcst_id = 90, omega_step = 5)
# norms = calculate_norms(_id[num], abt, fcst_density, lt, sl)
# norms[['stock', 'в пути', 'СЗ', 'норма', 'чистая норма','прогноз', 'квант норма', 'дата заказа']]\
#     .plot(title = 'product = {0} location = {1}<br>Service Level = {2}, PBR = {6}, Lead Time = {3}, \n Квант = {4}, \n МерчМинимум = {5}'\
#           .format(_id[num][0], _id[num][1],
#                   sl[(sl['product_id'] == _id[num][0])&(sl['location_id'] == _id[num][1])].SL.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].LT.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])]['Квант'].values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].MerchMinimum.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].PBR.values[0]))\
#     .update_layout(height=450, width=950)

In [ ]:
# horizon = 100
# num = 4
# fcst_density = build_density_forecast(abt, pair=_id[num], horizon = horizon, fcst_id = 90, omega_step = 1)
# norms = calculate_norms(_id[num], abt, fcst_density, lt, sl)
# norms[['stock', 'в пути', 'СЗ', 'норма', 'чистая норма','прогноз', 'квант норма', 'дата заказа']]\
#     .plot(title = 'product = {0} location = {1}<br>Service Level = {2}, PBR = {6}, Lead Time = {3}, \n Квант = {4}, \n МерчМинимум = {5}'\
#           .format(_id[num][0], _id[num][1],
#                   sl[(sl['product_id'] == _id[num][0])&(sl['location_id'] == _id[num][1])].SL.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].LT.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])]['Квант'].values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].MerchMinimum.values[0],
#                   lt[(lt['product_id'] == _id[num][0])&(lt['location_id'] == _id[num][1])].PBR.values[0]))\
#     .update_layout(height=450, width=950)

In [ ]:
# px.bar(dict_to_df(new_df), x = 'bins', y = 'probs', title = 'чистая норма = {0}'\
#        .format(get_percentile(reshape_barchart(new_df, step=1), _sl, )))\
#     .update_layout(height=300, width=900).show()

In [ ]:
# px.bar(dict_to_df(reshape_barchart(new_df, step=_quant)), x = 'bins', y = 'probs', title = 'квант_норма = {0}'\
#     .format(get_percentile(reshape_barchart(new_df, step=_quant), _sl)))\
#     .update_layout(height=300, width=900).show()

In [ ]:
# norms.ffill().bfill()[['норма', 'квант норма']]\
#     .plot(title = 'product = {0} location = {1}, \n Service Level = {2}, \n Lead Time = {3}, \n Квант = {4}, \n МерчМинимум = {5}'\
#           .format(pair[0], pair[1], _sl, _lt, _quant, _mm))\
#     .update_layout(height=450, width=950)

In [ ]:
# norms.transpose()[2:]
# norms[norms.columns[2:]][20:30]